# Explore the features - Phases 5 & 7

Geometric measurements, the width/depth profiles, the midline curve, and the
cohort feature table.

**Needs real captures in `data/raw/`**, and an axis convention saved by
`python -m src.normalization detect --save`.

Nothing here encodes a belief about attractiveness - these are measurements.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # run from notebooks/

import numpy as np, pandas as pd, matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.config import RAW_DIR
from src.data_loader import load_capture, find_capture_files
from src.normalization import normalize_capture, default_axis_convention
from src.landmarks import LandmarkSet

paths = find_capture_files(RAW_DIR)
assert paths, "No captures. Export from the iOS app into data/raw/ first."

convention = default_axis_convention()
assert convention is not None, "Run: python -m src.normalization detect --save"

capture = load_capture(paths[0])
face = normalize_capture(capture, convention)
print(face.key, "| side resolved:", face.side_resolved)

## 1. Features for one capture

Grouped by availability tier. `NaN` always carries a stated reason.

In [ ]:
from src.features import extract_features

fs = extract_features(face, LandmarkSet.load())
print(f"{len(fs.values)} features, {fs.n_missing} unavailable\n")

by_tier = {}
for name, src in fs.sources.items():
    by_tier.setdefault(src.split(":")[0], []).append(name)

for tier in sorted(by_tier):
    print(f"-- {tier} " + "-" * (52 - len(tier)))
    for n in by_tier[tier]:
        v = fs.values[n]
        print(f"   {n:34s} {v:12.6f}" if np.isfinite(v) else f"   {n:34s} {'NaN':>12s}")
    print()

In [ ]:
if fs.missing_reasons:
    print("Why features are unavailable:\n")
    for reason in sorted(set(fs.missing_reasons.values())):
        names = [n for n, r in fs.missing_reasons.items() if r == reason]
        more = " ..." if len(names) > 5 else ""
        print(f"  {reason}")
        print(f"    -> {len(names)} feature(s): {', '.join(names[:5])}{more}\n")

## 2. Width and depth profiles

Landmark-free stand-ins for "jaw width", "cheekbone width" and so on: the face's
extent at a height defined by a *proportion of the face*, not by an anatomical
point. Band 0 is inferior (chin), band 8 superior (forehead).

In [ ]:
from src.features import width_profile, depth_profile

w = width_profile(face.vertices)
d = depth_profile(face.vertices)
bands = np.arange(len(w))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].barh(bands, w, color="#4a7ba7")
axes[0].set_title("width profile")
axes[0].set_ylabel("band (0 = chin -> 8 = forehead)")
axes[0].set_xlabel("left-right extent (normalised)")
axes[1].barh(bands, d, color="#a7744a")
axes[1].set_title("depth profile (max anterior projection)")
axes[1].set_xlabel("anterior extent (normalised)")
for a in axes:
    a.set_yticks(bands)
plt.tight_layout(); plt.show()

## 3. The midline curve

Where `nose_tip`, `nasion`, `subnasale`, `pogonion` and `menton` come from -
extrema of this curve, by the rules in `src/features.py::MidlineLandmarks`.

A point with no clear extremum is reported **not found**, never back-filled.

In [ ]:
from src.features import midline_curve, derive_midline_landmarks

y, z = midline_curve(face.vertices)
ml = derive_midline_landmarks(face.vertices)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(z, y, "-", color="#444", lw=1.5, label="midsagittal profile")
for name, colour in [("nose_tip", "crimson"), ("nasion", "seagreen"),
                     ("subnasale", "darkorange"), ("pogonion", "royalblue"),
                     ("menton", "purple")]:
    p = getattr(ml, name)
    if p is None:
        print(f"  {name}: NOT FOUND")
        continue
    ax.plot(p[2], p[1], "o", color=colour, ms=9)
    ax.annotate(f" {name}", (p[2], p[1]), color=colour, fontsize=9)
ax.set_xlabel("anterior (+Z)")
ax.set_ylabel("superior (+Y)")
ax.set_title("Midline curve and derived points")
ax.set_aspect("equal")
plt.tight_layout(); plt.show()
for n in ml.notes:
    print("note:", n)

## 4. The cohort table

`src.dataset` writes `data/processed/features.csv`. Check coverage before
modelling: entirely-missing and zero-variance columns carry no information.

In [ ]:
from src.dataset import build_feature_table, describe_table, save_feature_table

table, problems = build_feature_table(RAW_DIR)
for p in problems:
    print("skipped:", p)
print(describe_table(table))

In [ ]:
display(table.head())
save_feature_table(table)

## 5. Feature distributions across the cohort

In [ ]:
from src.dataset import feature_columns

cols = [c for c in feature_columns(table) if table[c].notna().all() and table[c].nunique() > 1]
print(f"{len(cols)} usable feature(s)")

if len(table) < 5:
    print("Too few captures for distributions to mean anything yet.")
else:
    show = cols[:12]
    fig, axes = plt.subplots(3, 4, figsize=(14, 8))
    for ax, c in zip(axes.ravel(), show):
        ax.hist(table[c].dropna(), bins=min(20, max(5, len(table) // 2)), color="#4a7ba7")
        ax.set_title(c, fontsize=8)
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[len(show):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

## 6. Correlation between features

Facial measurements are heavily intercorrelated. This matters directly for
interpretability later: where two features carry the same information, a model
picks one arbitrarily and the other looks unimportant. See `CAUSAL_DISCLAIMER`
in `src/evaluate.py`.

In [ ]:
if len(table) < 10:
    print(f"Only {len(table)} capture(s) - correlations need more data to be meaningful.")
else:
    corr = table[cols].corr()
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=90, fontsize=6)
    ax.set_yticks(range(len(cols)))
    ax.set_yticklabels(cols, fontsize=6)
    fig.colorbar(im, label="Pearson r")
    ax.set_title("Feature correlation")
    plt.tight_layout(); plt.show()

    pairs = (corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
                 .stack().sort_values(key=abs, ascending=False))
    print("Most correlated pairs:")
    for (a, b), r in pairs.head(10).items():
        print(f"  {r:+.3f}  {a} / {b}")

---
## Next

Collect human ratings, then:

```bash
python -m src.ratings aggregate     # check the ICC ceiling before modelling
python -m src.train --save
```

`src/train.py` will not run without real labels, by design.